# Azure AI Search Hybrid RAG — Python उदाहरण

यह notebook आपके मौजूदा Azure OpenAI deployment और Azure AI Search index के साथ एक छोटा RAG flow चलाता है: प्रश्न की embedding, keyword + vector hybrid search, और retrieved स्रोतों पर आधारित उत्तर। Azure AI Portal में Azure OpenAI और Azure AI Search resources बनाकर deployments/index तैयार करें।

यह project के `.env` variable names और `content` / `content_vector` fields का उपयोग करता है। Search index पहले से बना होना चाहिए; notebook index schema नहीं बदलता।

## 1. आवश्यक packages और configuration

Portal से resource endpoint/key और deployment names लेकर उन्हें project की `.env` file में रखें। इस notebook में keys कभी print नहीं की जातीं। Search index में `id` key field, searchable `content` field, और embedding model के dimensions से मेल खाता `content_vector` vector field होना चाहिए।

In [1]:
%pip install -q openai azure-search-documents azure-core python-dotenv pypdf


[notice] A new release of pip is available: 26.0 -> 26.3
[notice] To update, run: /Users/admin/Downloads/azure-rag-project/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery

load_dotenv(override=True)

required_settings = [
    "AZURE_OPENAI_BASE_URL",
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_EMBEDDING_DEPLOYMENT",
    "AZURE_OPENAI_CHAT_DEPLOYMENT",
    "AZURE_SEARCH_ENDPOINT",
    "AZURE_SEARCH_KEY",
    "AZURE_SEARCH_INDEX",
]
missing_settings = [name for name in required_settings if not os.getenv(name)]
if missing_settings:
    raise ValueError(f".env में आवश्यक settings नहीं मिलीं: {', '.join(missing_settings)}")

openai_client = OpenAI(
    base_url=os.environ["AZURE_OPENAI_BASE_URL"].rstrip("/") + "/",
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
)
search_client = SearchClient(
    endpoint=os.environ["AZURE_SEARCH_ENDPOINT"],
    index_name=os.environ["AZURE_SEARCH_INDEX"],
    credential=AzureKeyCredential(os.environ["AZURE_SEARCH_KEY"]),
)

EMBEDDING_DEPLOYMENT = os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"]
CHAT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"]
VECTOR_FIELD = os.getenv("AZURE_SEARCH_VECTOR_FIELD", "content_vector")
print("Azure clients तैयार हैं। Index:", os.environ["AZURE_SEARCH_INDEX"])
print("Index में documents:", search_client.get_document_count())

Azure clients तैयार हैं। Index: airbnb-rag-v1
Index में documents: 43


## 2. Document helpers

`get_document` को index में मौजूद document की key (`id`) दें। `get_document_count()` index में documents की कुल संख्या देता है।

In [3]:
def get_document(document_key: str):
    """Search index से key के आधार पर एक document पढ़ें."""
    return search_client.get_document(key=document_key)


def compute_text_embedding(text: str) -> list[float]:
    """Azure OpenAI embedding deployment से text का vector बनाएँ."""
    response = openai_client.embeddings.create(
        model=EMBEDDING_DEPLOYMENT,
        input=text,
    )
    return response.data[0].embedding


print("Document count:", search_client.get_document_count())
# उदाहरण: पहले से ज्ञात id के लिए चलाएँ: get_document("your-document-id")

Document count: 43


## 3. Airbnb PDF को index करना

यह project के `downloads/adbf1dfa3b5b_airbnb_listings.pdf` से हर listing पढ़ता है, embeddings बनाता है, और stable IDs के साथ Azure AI Search में merge-or-upload करता है। दोबारा चलाने पर duplicate listings नहीं बनेंगी।

In [4]:
import json
import re
import time
from pathlib import Path

from pypdf import PdfReader

pdf_path = Path("downloads/adbf1dfa3b5b_airbnb_listings.pdf")
if not pdf_path.is_file():
    raise FileNotFoundError(f"PDF नहीं मिली: {pdf_path.resolve()}")

listings = []
for page_number, page in enumerate(PdfReader(pdf_path).pages, start=1):
    page_text = (page.extract_text() or "").strip()
    match = re.search(
        r"Property ID:\s*(\S+)\s*\n([^\n]+)\s*\nSUMMARY\s*\n(.*)",
        page_text,
        re.DOTALL,
    )
    if match is None:
        raise ValueError(f"Page {page_number} has no parseable listing record")
    property_id, title, summary = match.groups()
    content = f"{title.strip()}\n\n{summary.strip()}"
    listings.append({
        "id": f"listing-{property_id}",
        "content": content,
        "metadata": json.dumps({
            "source": str(pdf_path),
            "page": page_number,
            "property_id": property_id,
            "title": title.strip(),
        }),
    })

if not listings or len({listing["id"] for listing in listings}) != len(listings):
    raise ValueError("PDF listings are empty or contain duplicate property IDs")

embedding_response = openai_client.embeddings.create(
    model=EMBEDDING_DEPLOYMENT,
    input=[listing["content"] for listing in listings],
)
if len(embedding_response.data) != len(listings):
    raise RuntimeError("Embedding response count does not match listing count")
for item in embedding_response.data:
    listings[item.index]["content_vector"] = item.embedding

upload_results = search_client.merge_or_upload_documents(documents=listings)
if len(upload_results) != len(listings):
    raise RuntimeError("Search did not return a result for every listing")
failed_uploads = [result for result in upload_results if not result.succeeded]
if failed_uploads:
    raise RuntimeError(
        "Listing upload failed: "
        + "; ".join(
            f"{result.key}: {result.error_message or 'Unknown Search error'}"
            for result in failed_uploads
        )
    )

indexed_count = search_client.get_document_count()
for _ in range(15):
    if indexed_count >= len(listings):
        break
    time.sleep(2)
    indexed_count = search_client.get_document_count()
if indexed_count < len(listings):
    raise RuntimeError(
        f"Only {indexed_count} documents are visible; expected at least {len(listings)}"
    )
print(f"Uploaded {len(upload_results)} listings; index now contains {indexed_count} documents.")

Uploaded 43 listings; index now contains 43 documents.


## 3. Hybrid search

यह keyword query (`search_text`) और vector query (`vector_queries`) दोनों भेजता है। `top=5` से अधिकतम पाँच matching documents मिलेंगे।

In [5]:
def search_documents(question: str, top: int = 5) -> list[dict]:
    if not question.strip():
        raise ValueError("प्रश्न खाली नहीं होना चाहिए।")
    if top < 1:
        raise ValueError("top कम से कम 1 होना चाहिए।")

    question_vector = compute_text_embedding(question)
    vector_query = VectorizedQuery(
        vector=question_vector,
        k_nearest_neighbors=top,
        fields=VECTOR_FIELD,
    )
    results = search_client.search(
        search_text=question,
        vector_queries=[vector_query],
        top=top,
        select=["id", "content", "metadata"],
    )
    return [dict(result) for result in results]


question = "Which listings have a garden or city views?"
documents = search_documents(question, top=5)
print(f"Retrieved {len(documents)} documents")
for document in documents:
    print("-", document.get("id"), str(document.get("content", ""))[:180])

Retrieved 5 documents
- listing-958 Bright, Modern Garden Unit - 1BR/1B

New update: the house next door is under construction and there is the possibility of noise from 7am - 5pm.
Our rates are discounted during thi
- listing-23611 Bernal Heights Gem with Fab Views !

This listing is for a very cozy & comfortable spare guest room in a lovely 2 bedroom, 1 bath condo in
fantastic Bernal Heights. The bedroom con
- listing-12584 3 levels, Views, 2BR/1BA

Unique property. Unforgettable views, quiet yet very convenient location. Vaulted ceilings and back yard.
- listing-10820 Haight Buena Vista Park Garden 3BR

This property is only available with a minimum stay of one month. This historic 3 bedroom Victorian house
sits on a hillside overlooking the Cit
- listing-10832 Union Square Modern Loft

This property is only available with a minimum stay of one month. Brand new residential building
incorporating touches of the City's rich architectural hi


## 4. Grounded answer और citations

Retrieved text को source labels के साथ model के context में भेजा जाता है। System prompt निर्देश देता है कि उत्तर context पर आधारित हो और जानकारी न मिलने पर स्पष्ट रूप से बताए।

In [6]:
def generate_answer(question: str, top: int = 5) -> str:
    documents = search_documents(question, top=top)
    if not documents:
        return "Search index में इस प्रश्न के लिए कोई स्रोत नहीं मिला।"

    sources = "\n\n".join(
        f"[Source {index}] (document id: {document.get('id', 'unknown')})\n"
        f"{document.get('content', '')}"
        for index, document in enumerate(documents, start=1)
    )
    response = openai_client.chat.completions.create(
        model=CHAT_DEPLOYMENT,
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer using only the supplied sources. If the sources do not "
                    "contain the answer, say you do not know. Cite supporting claims "
                    "with source labels such as [Source 1]."
                ),
            },
            {
                "role": "user",
                "content": f"Question: {question}\n\nSources:\n{sources}",
            },
        ],
    )
    return response.choices[0].message.content or "कोई उत्तर नहीं मिला।"


print(generate_answer(question))

The listings that have a garden or city views are:

- Listing-958: Bright, Modern Garden Unit - 1BR/1B. This garden unit overlooks a grassy backyard area with fruit trees and native plants [Source 1].
- Listing-12584: 3 levels, Views, 2BR/1BA. This property has unforgettable views and a backyard [Source 3].
- Listing-10820: Haight Buena Vista Park Garden 3BR. This house has a great garden and overlooks the City and Marin Headlands from a large private roof-top deck [Source 4].

Listing-23611 mentions a view of Bernal Hill but does not specify garden or city views explicitly [Source 2]. Listing-10832 does not mention garden or city views [Source 5].


## 5. एक document upload करने का helper

नीचे का helper एक document का vector बनाकर Azure AI Search में लिखता है। Airbnb PDF की सभी listings को batch में और idempotently index करने के लिए अगला section चलाएँ। `id`, `content`, `content_vector`, और `metadata` fields index schema से मेल खाने चाहिए।

In [7]:
def upload_document(document_id: str, content: str, metadata: str = "{}"):
    if not document_id.strip() or not content.strip():
        raise ValueError("document_id और content खाली नहीं होने चाहिए।")
    document = {
        "id": document_id,
        "content": content,
        "content_vector": compute_text_embedding(content),
        "metadata": metadata,
    }
    results = search_client.merge_or_upload_documents(documents=[document])
    failed = [result for result in results if not result.succeeded]
    if failed:
        raise RuntimeError(
            "Document upload failed: "
            + "; ".join(result.error_message or "Unknown Search error" for result in failed)
        )
    return results


# Example: upload_document("listing-958", "Listing text", '{"property_id": "958"}')